# 🏆 학생 건강상태 분류 — 앙상블 실험

## 실험 목적

앞선 모델링 단계에서 단일 모델의 성능을 확인한 뒤, 여러 부스팅 모델을 결합했을 때 성능이 개선되는지 실험한다.

현재 단일 모델의 점수가 약 0.904 수준이므로, 앙상블을 적용해 성능과 예측의 안정성이 좋아지는지 확인하는 것이 목표다.

이번 실험은 앞선 EDA와 모델 비교 과정을 대체하지 않고, 최종 단계에서 추가로 진행하는 성능 개선 실험으로 정리한다.

## 실험 내용

### 1. OOF 교차검증

각 Fold에서 학습에 사용되지 않은 데이터에 대해 예측하는 OOF(Out-Of-Fold) 방식을 사용한다.

이를 통해 학습에 사용한 데이터를 다시 평가하는 문제를 줄이고, 앙상블 성능을 비교할 때 보다 현실적인 점수를 확인한다.

### 2. 부스팅 모델 앙상블

다음 4개 모델을 사용한다.

- LightGBM
- XGBoost
- CatBoost
- HistGradientBoosting

모델마다 데이터를 나누는 방식과 학습하는 패턴이 조금씩 다르기 때문에, 서로 다른 예측 결과를 만들 수 있다. 이를 이용해 각 모델의 예측 확률을 결합한다.

### 3. Fold별 모델 평균

각 Fold에서 학습한 모델로 Test 데이터를 예측한 뒤, Fold별 예측 확률을 평균낸다.

한 번 학습한 모델의 결과만 사용하는 것보다 여러 Fold의 결과를 평균내면 특정 데이터 분할에 따른 영향을 줄일 수 있다.

### 4. Early Stopping

부스팅 모델의 트리 개수를 미리 크게 정하지 않고, Validation 성능이 더 이상 좋아지지 않을 때 학습을 멈추도록 설정한다.

이를 통해 필요 이상으로 학습하는 것을 줄이고 과적합 가능성을 확인한다.

## 앙상블을 사용하는 이유

각 모델의 Validation 점수가 비슷하더라도 잘못 예측하는 데이터가 완전히 같지는 않을 수 있다.

예측 확률을 평균내면 한 모델이 틀린 샘플을 다른 모델이 보완할 가능성이 있다. 다만 앙상블이 항상 성능을 높여주는 것은 아니므로, 단일 모델과 동일한 조건에서 성능을 비교한 뒤 적용 여부를 결정한다.

## 비교할 결과

앙상블 적용 전후에 다음 결과를 비교한다.

- 단일 최종 모델의 Balanced Accuracy
- 4개 모델 앙상블의 Balanced Accuracy
- OOF 교차검증 평균과 표준편차
- 클래스별 Recall
- Confusion Matrix
- Kaggle 제출 점수

앙상블 성능이 단일 모델보다 높지 않거나 특정 클래스의 Recall이 크게 떨어지는 경우에는, 앙상블을 최종 모델로 사용하지 않고 단일 모델을 유지한다.

> 이번 단계의 목표는 앙상블 자체를 사용하는 것이 아니라, 실제 검증 결과를 바탕으로 앙상블이 도움이 되는지 확인하는 것이다.

## 0. 설치 & 설정

**핵심 설정(맨 위에서 조절):**
- `METRIC` / `USE_CLASS_WEIGHT`: 대회 지표에 맞춤. **balanced accuracy면 True 유지**, 혹시 **그냥 accuracy면 False**로.
- `RUN_FRACTION`: 처음엔 0.3으로 빠르게 파이프라인 검증 → 잘 돌면 **1.0으로 바꿔 최종 실행**.


In [1]:
!pip install -q lightgbm xgboost catboost
print('설치 완료')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00
설치 완료


In [2]:
import time, warnings, os
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

# ===== 핵심 설정 =====
METRIC           = 'balanced_accuracy'   # 대회 지표. 'accuracy'이면 아래 USE_CLASS_WEIGHT=False 권장
USE_CLASS_WEIGHT = True                   # balanced accuracy면 True, 순수 accuracy면 False
RUN_FRACTION     = 0.3                    # 먼저 0.3으로 검증 → 최종엔 1.0
N_SPLITS         = 5
RS               = 42
np.random.seed(RS)
score_fn = balanced_accuracy_score if METRIC=='balanced_accuracy' else accuracy_score
print(f'METRIC={METRIC} | class_weight={USE_CLASS_WEIGHT} | fraction={RUN_FRACTION} | folds={N_SPLITS}')


METRIC=balanced_accuracy | class_weight=True | fraction=0.3 | folds=5


## 1. 데이터 로드 + 파생변수 (self-contained)

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
DATA_PATH='/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/kaggle_student_classification/'
train=pd.read_csv(DATA_PATH+'train.csv'); test=pd.read_csv(DATA_PATH+'test.csv')
TARGET,ID='health_condition','id'
numeric_features=['sleep_duration','heart_rate','bmi','calorie_expenditure','step_count','exercise_duration','water_intake']
categorical_features=['diet_type','stress_level','sleep_quality','physical_activity_level','smoking_alcohol','gender']
feature_cols=numeric_features+categorical_features
HR_HI=train['heart_rate'].quantile(0.75); STEP_LO=train['step_count'].quantile(0.25); WATER_LO=train['water_intake'].quantile(0.25)
ord_maps={'stress_level':{'low':0,'medium':1,'high':2},'sleep_quality':{'poor':0,'average':1,'good':2},
 'physical_activity_level':{'sedentary':0,'moderate':1,'active':2},'smoking_alcohol':{'no':0,'occasional':1,'yes':2}}
def make_features(df):
    X=df.copy()
    for col,m in ord_maps.items(): X[col+'_ord']=X[col].map(m)
    X['sleep_debt']=(7-X['sleep_duration']).clip(lower=0); X['sleep_excess']=(X['sleep_duration']-9).clip(lower=0)
    X['sleep_ideal']=X['sleep_duration'].between(7,9).astype('float'); X.loc[X['sleep_duration'].isna(),'sleep_ideal']=np.nan
    X['bmi_cat']=pd.cut(X['bmi'],bins=[-np.inf,18.5,25,30,np.inf],labels=[0,1,2,3]).astype('float')
    X['bmi_abnormal']=(~X['bmi'].between(18.5,25)).astype('float'); X.loc[X['bmi'].isna(),'bmi_abnormal']=np.nan
    X['hr_high']=(X['heart_rate']>HR_HI).astype('float'); X.loc[X['heart_rate'].isna(),'hr_high']=np.nan
    X['steps_per_ex_min']=X['step_count']/(X['exercise_duration']+1); X['cal_per_step']=X['calorie_expenditure']/(X['step_count']+1)
    X['low_activity']=(X['step_count']<STEP_LO).astype('float'); X.loc[X['step_count'].isna(),'low_activity']=np.nan
    X['low_water']=(X['water_intake']<WATER_LO).astype('float'); X.loc[X['water_intake'].isna(),'low_water']=np.nan
    risk=pd.DataFrame(index=X.index)
    risk['a']=(X['sleep_duration']<6).astype(float); risk['b']=(X['stress_level']=='high').astype(float)
    risk['c']=(X['sleep_quality']=='poor').astype(float); risk['d']=(X['physical_activity_level']=='sedentary').astype(float)
    risk['e']=(X['step_count']<STEP_LO).astype(float); risk['f']=(X['smoking_alcohol']=='yes').astype(float)
    risk['g']=(~X['bmi'].between(18.5,25)).astype(float)
    X['lifestyle_risk_score']=risk.sum(axis=1); X['n_missing']=df[feature_cols].isnull().sum(axis=1)
    for c in feature_cols: X[c+'_isna']=df[c].isnull().astype('int8')
    return X
train_fe=make_features(train); test_fe=make_features(test)

# 명목형(diet_type, gender)을 숫자 코드로 (트리는 NaN 그대로 처리 가능)
for c in ['diet_type','gender']:
    cats=train_fe[c].astype('category').cat.categories
    mp={k:i for i,k in enumerate(cats)}
    train_fe[c+'_code']=train_fe[c].map(mp); test_fe[c+'_code']=test_fe[c].map(mp)
print('train_fe', train_fe.shape)


train_fe (690088, 46)


## 2. 두 가지 변수 세트 비교

변수 가지치기가 실제 성능에 영향을 주는지 확인하기 위해 두 가지 변수 세트를 같은 조건에서 비교한다.

### 비교할 변수 세트

- `FEATURES_PRUNED`
  - 02번 모델링 노트북에서 정리한 최종 변수 세트
  - MI 결과와 앞선 검증을 바탕으로 관련성이 낮은 변수를 제외한 구성

- `FEATURES_FULL`
  - 파생변수와 결측 여부 변수까지 모두 포함한 전체 변수 세트
  - 가지치기 전 상태와 비교하기 위한 기준으로 사용

### 비교 목적

EDA 단계에서 확인한 빠른 테스트 점수는 약 0.947이었지만, 실제 모델링 과정에서는 약 0.904 수준으로 낮아졌다.

이 차이가 변수 구성 때문인지, 아니면 데이터 분할이나 검증 방식의 차이 때문인지 확인하기 위해 두 변수 세트를 동일한 조건에서 비교한다.

두 실험은 다음 조건을 동일하게 유지한다.

- 같은 Train/Validation 분할
- 같은 random seed
- 같은 모델과 하이퍼파라미터
- 같은 평가 지표
- 같은 학습 데이터

### 전처리 방식

이번 비교에 사용하는 부스팅 모델은 결측치를 자체적으로 처리할 수 있으므로, 별도의 대치나 스케일링을 적용하지 않는다.

- 결측치 대치: 적용하지 않음
- 표준화·정규화: 적용하지 않음
- 부스팅 모델이 결측치를 직접 처리

두 변수 세트의 성능 차이를 확인한 뒤, 변수 가지치기가 실제로 도움이 되었는지 판단한다.

> 변수를 줄였을 때 성능이 좋아지는지뿐만 아니라, 성능 차이가 거의 없으면서 변수 수와 해석 복잡도만 줄어드는지도 함께 확인한다.

In [5]:
FEATURES_PRUNED = numeric_features + [
    'lifestyle_risk_score','sleep_debt','sleep_ideal','bmi_cat','cal_per_step','steps_per_ex_min','low_activity',
    'stress_level_ord','sleep_quality_ord','physical_activity_level_ord','smoking_alcohol_ord',
    'diet_type_code','gender_code']
exclude = [ID, TARGET] + categorical_features
FEATURES_FULL = [c for c in train_fe.columns if c not in exclude and train_fe[c].dtype != 'object']
print('PRUNED', len(FEATURES_PRUNED), '| FULL', len(FEATURES_FULL))

# 라벨 인코딩 + 실행 샘플
le = LabelEncoder().fit(train_fe[TARGET])
CLASSES = le.classes_
if RUN_FRACTION < 1.0:
    run = train_fe.groupby(TARGET, group_keys=False).apply(lambda s: s.sample(frac=RUN_FRACTION, random_state=RS))
else:
    run = train_fe
y_run = le.transform(run[TARGET])
print('실행 데이터:', run.shape, '| 클래스:', dict(zip(CLASSES, np.bincount(y_run))))


PRUNED 20 | FULL 38
실행 데이터: (207026, 46) | 클래스: {'at-risk': np.int64(177768), 'fit': np.int64(11941), 'unhealthy': np.int64(17317)}


## 3. OOF 학습 함수 (핵심 엔진)

한 모델을 K-fold로 학습하면서:
- 각 fold의 검증부분 확률 → **OOF 예측**(정직한 CV 점수 계산용)
- 각 fold 모델의 **test 확률을 평균** → 배깅된 test 예측
- **early stopping**으로 나무 수 자동 결정

네 모델(LGBM/XGB/CatBoost/HGB)의 early stopping API가 조금씩 달라 분기 처리


In [6]:
def make_model(name):
    if name=='LightGBM':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=63,
                              subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                              random_state=RS, n_jobs=-1, verbose=-1)
    if name=='XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=3000, learning_rate=0.03, max_depth=6,
                             subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist',
                             early_stopping_rounds=100, eval_metric='mlogloss',
                             random_state=RS, n_jobs=-1, verbosity=0)
    if name=='CatBoost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(iterations=3000, learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
                                  random_state=RS, verbose=0)
    if name=='HistGB':
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=3000, learning_rate=0.03, max_leaf_nodes=63,
                                              l2_regularization=1.0, early_stopping=True,
                                              validation_fraction=0.1, n_iter_no_change=100, random_state=RS)

def fit_one(name, model, Xtr, ytr, Xva, yva, sw):
    if name=='LightGBM':
        from lightgbm import early_stopping, log_evaluation
        model.fit(Xtr, ytr, sample_weight=sw, eval_set=[(Xva,yva)],
                  callbacks=[early_stopping(100, verbose=False), log_evaluation(0)])
    elif name=='XGBoost':
        model.fit(Xtr, ytr, sample_weight=sw, eval_set=[(Xva,yva)], verbose=False)
    elif name=='CatBoost':
        model.fit(Xtr, ytr, sample_weight=sw, eval_set=(Xva,yva), early_stopping_rounds=100, verbose=0)
    else:  # HistGB: 내장 early stopping
        model.fit(Xtr, ytr, sample_weight=sw)
    return model

def run_oof(name, features):
    X = run[features]; Xtest = test_fe[features]
    n_cls = len(CLASSES)
    oof = np.zeros((len(X), n_cls)); test_proba = np.zeros((len(Xtest), n_cls))
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=RS)
    t0 = time.time()
    for fold,(tr,va) in enumerate(skf.split(X, y_run)):
        Xtr,Xva = X.iloc[tr], X.iloc[va]; ytr,yva = y_run[tr], y_run[va]
        sw = compute_sample_weight('balanced', ytr) if USE_CLASS_WEIGHT else None
        m = fit_one(name, make_model(name), Xtr, ytr, Xva, yva, sw)
        oof[va] = m.predict_proba(Xva)
        test_proba += m.predict_proba(Xtest) / N_SPLITS
    sc = score_fn(y_run, oof.argmax(1))
    print(f'  {name:10s} OOF {METRIC}={sc:.4f}  ({time.time()-t0:.0f}s)')
    return oof, test_proba, sc


## 4. 가지치기 진단 (PRUNED vs FULL)

LightGBM 하나로 두 세트를 OOF 비교 **점수 높은 쪽을 최종 변수 세트로 채택.**
(0.947 vs 0.904 격차가 변수 때문인지 여기서 판가름)


In [7]:
print('▶ 가지치기 진단 (LightGBM OOF)')
diag = {}
for tag, feats in [('PRUNED', FEATURES_PRUNED), ('FULL', FEATURES_FULL)]:
    _,_,sc = run_oof('LightGBM', feats); diag[tag]=sc
BEST_FEATURES = FEATURES_FULL if diag['FULL'] >= diag['PRUNED'] else FEATURES_PRUNED
best_tag = 'FULL' if diag['FULL']>=diag['PRUNED'] else 'PRUNED'
print(f"\n채택: {best_tag}  (PRUNED={diag['PRUNED']:.4f} vs FULL={diag['FULL']:.4f})")
print('→ 두 값이 비슷하면 변수 탓이 아니라 이전의 소표본 낙관이었다는 뜻. 큰 차이면 가지치기가 원인.')


▶ 가지치기 진단 (LightGBM OOF)
  LightGBM   OOF balanced_accuracy=0.9208  (2397s)
  LightGBM   OOF balanced_accuracy=0.9221  (2211s)

채택: FULL  (PRUNED=0.9208 vs FULL=0.9221)
→ 두 값이 비슷하면 변수 탓이 아니라 이전의 소표본 낙관이었다는 뜻. 큰 차이면 가지치기가 원인.


## 5. ⭐ 4종 부스팅 모델 학습과 앙상블

앞 단계에서 비교한 결과를 바탕으로 최종 변수 세트를 정한 뒤, 4개의 부스팅 모델을 각각 학습한다.

이번 단계에서는 단일 모델의 예측만 사용하지 않고, 각 모델의 Test 예측 확률을 평균내어 앙상블 결과를 만든다.

### 사용할 모델

- LightGBM
- XGBoost
- CatBoost
- HistGradientBoosting

### 학습 방법

각 모델은 동일한 Fold 구성으로 OOF 교차검증을 진행한다.

1. Train 데이터를 Fold별로 나눈다.
2. 일부 Fold로 모델을 학습한다.
3. 학습에 사용하지 않은 Fold의 Validation 데이터를 예측한다.
4. 모든 Fold의 Validation 예측을 모아 OOF 예측을 만든다.
5. 각 Fold에서 학습한 모델로 Test 데이터를 예측한다.
6. 모델별 Test 예측 확률을 평균내어 최종 앙상블 확률을 계산한다.

OOF 예측은 모델이 학습에 사용하지 않은 데이터에 대한 예측을 모아 평가하는 방식이다. 이를 사용하면 학습 데이터에 대한 예측으로 성능을 확인할 때보다 과적합 여부를 확인하기 쉽다.

### 실행 시간 관리

부스팅 모델 4개를 여러 Fold로 학습하기 때문에 단일 모델보다 실행 시간이 길 수 있다.

먼저 일부 데이터로 코드가 정상적으로 동작하는지 확인한 뒤, 최종 실행에서 전체 데이터를 사용한다.

```python
RUN_FRACTION = 0.3  # 코드 테스트용

# 최종 실행 시
RUN_FRACTION = 1.0  # 전체 데이터 사용


RUN_FRACTION = 0.3: 코드와 결과 형식 확인
RUN_FRACTION = 1.0: 전체 데이터로 최종 학습

In [8]:
MODELS = ['LightGBM','XGBoost','CatBoost','HistGB']
oofs, tests, scores = {}, {}, {}
print('▶ 4종 OOF 학습')
for name in MODELS:
    try:
        o,t,s = run_oof(name, BEST_FEATURES)
        oofs[name],tests[name],scores[name] = o,t,s
    except Exception as e:
        print(f'  {name} 제외: {str(e)[:70]}')

# 앙상블: OOF 확률 평균 → 점수 / test 확률 평균 → 제출용
ok = list(oofs.keys())
oof_ens  = np.mean([oofs[n]  for n in ok], axis=0)
test_ens = np.mean([tests[n] for n in ok], axis=0)
ens_score = score_fn(y_run, oof_ens.argmax(1))

tbl = pd.DataFrame({'model':ok+['🏆 앙상블'],
                    METRIC:[scores[n] for n in ok]+[ens_score]}).sort_values(METRIC, ascending=False)
display(tbl.round(4))
print(f'\n앙상블 {METRIC} = {ens_score:.4f}  (단일 최고 = {max(scores.values()):.4f})')


▶ 4종 OOF 학습
  LightGBM   OOF balanced_accuracy=0.9221  (2249s)
  XGBoost    OOF balanced_accuracy=0.9259  (3550s)
  CatBoost   OOF balanced_accuracy=0.9485  (2072s)
  HistGB     OOF balanced_accuracy=0.9480  (423s)


,model,balanced_accuracy
2,CatBoost,0.9485
3,HistGB,0.9480
4,🏆 앙상블,0.9426
1,XGBoost,0.9259
0,LightGBM,0.9221



앙상블 balanced_accuracy = 0.9426  (단일 최고 = 0.9485)


In [9]:
# 앙상블 상세 리포트 (OOF 기준)
print(classification_report(y_run, oof_ens.argmax(1), target_names=CLASSES))


              precision    recall  f1-score   support

     at-risk       0.99      0.95      0.97    177768
         fit       0.77      0.94      0.85     11941
   unhealthy       0.76      0.94      0.84     17317

    accuracy                           0.95    207026
   macro avg       0.84      0.94      0.89    207026
weighted avg       0.96      0.95      0.95    207026



## 6. 제출 파일 생성

앙상블의 배깅된 test 확률 → argmax → 라벨 복원. (fold 배깅이 이미 반영됨)


In [10]:
pred = le.inverse_transform(test_ens.argmax(1))
submission = pd.DataFrame({ID: test[ID], TARGET: pred})
tag = 'full' if RUN_FRACTION>=1.0 else f'frac{RUN_FRACTION}'
path = DATA_PATH + f'submission_ensemble_{tag}.csv'
submission.to_csv(path, index=False)
print('저장:', path, submission.shape)
print(submission[TARGET].value_counts(normalize=True).round(3).to_dict())
submission.head()


저장: /content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/kaggle_student_classification/submission_ensemble_frac0.3.csv (295753, 2)
{'at-risk': 0.827, 'unhealthy': 0.103, 'fit': 0.07}


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## 7. (선택) 앙상블 가중치 조정

Balanced Accuracy를 기준으로 모델별 예측 확률에 서로 다른 가중치를 적용해 본다.

간단한 랜덤 서치로 OOF 성능이 가장 높은 가중치 조합을 찾고, 단순 평균 앙상블과 비교한다.

> 가중치 조정 후 성능이 실제로 좋아지는 경우에만 최종 예측에 적용

In [11]:
best_w, best_s = None, ens_score
rng = np.random.RandomState(RS)
for _ in range(3000):
    w = rng.dirichlet(np.ones(len(ok)))               # 합=1 랜덤 가중치
    blend = np.tensordot(w, np.array([oofs[n] for n in ok]), axes=(0,0))
    s = score_fn(y_run, blend.argmax(1))
    if s > best_s: best_s, best_w = s, w
if best_w is not None:
    print('가중치 탐색 결과:', {n:round(float(x),3) for n,x in zip(ok,best_w)}, f'| OOF {METRIC}={best_s:.4f} (+{best_s-ens_score:.4f})')
    test_ens_w = np.tensordot(best_w, np.array([tests[n] for n in ok]), axes=(0,0))
    sub_w = pd.DataFrame({ID: test[ID], TARGET: le.inverse_transform(test_ens_w.argmax(1))})
    sub_w.to_csv(DATA_PATH + f'submission_ensemble_weighted_{tag}.csv', index=False)
    print('가중 앙상블 저장 완료')
else:
    print('단순 평균이 이미 최적 (가중치 개선 없음)')


가중치 탐색 결과: {'LightGBM': 0.005, 'XGBoost': 0.018, 'CatBoost': 0.531, 'HistGB': 0.446} | OOF balanced_accuracy=0.9486 (+0.0060)
가중 앙상블 저장 완료


## ✅ 정리 및 다음 단계

이번 앙상블 실험에서는 OOF 교차검증으로 성능을 확인하고, 4개 부스팅 모델의 예측 확률을 평균내 최종 결과를 만들었다.

추가로 Fold별 모델을 평균내는 방식과 Early Stopping을 적용해 특정 데이터 분할이나 과적합의 영향을 줄이고자 했다. 필요하면 모델별 가중치를 조정해 단순 평균과 비교할 수 있다.

### 실행 순서

1. `RUN_FRACTION = 0.3`으로 코드와 전체 흐름을 먼저 확인
2. 오류가 없으면 `RUN_FRACTION = 1.0`으로 전체 데이터를 사용해 다시 학습
3. 생성된 `submission_ensemble_full.csv`를 제출해 단일 모델과 성능을 비교

앙상블 성능이 기대보다 낮다면 다음 실험으로 하이퍼파라미터 튜닝, 모델 조합 변경, 주요 변수 간 상호작용 추가 등을 순서대로 확인할 수 있다.

우선 전체 데이터로 앙상블을 실행하고, 실제 리더보드 점수를 확인한 뒤 다음 개선 방향을 결정